# Full CI

In [29]:
from src_live import Molecule, BasisSet, MolecularIntegrals, ELEMENT_SYMBOL, Shell, S, T, V, ERI
import numpy as np

In [30]:
H4_xyz = """4
H4 molecule
H 1.0 1.0 0.0
H 1.0 -1.0 0.0
H -1.0 1.0 0.0
H -1.0 -1.0 0.0
"""
H4 = Molecule.from_string(H4_xyz)
sto3g = BasisSet("sto-3g")
sto3g.download_from_bse(["H"])

Dump data to sto-3g.json


In [31]:
molints = MolecularIntegrals(H4, sto3g)
S_my = molints.overlap_matrix()
T_my = molints.kinetic_matrix()
V_my = molints.nuclear_attraction_matrix()
ERI_my = molints.electron_repulsion_tensor(symmetrize = True)

In [32]:
from scipy.sparse import kron, csc_matrix, eye
from itertools import combinations
from scipy.sparse.linalg import eigs

class ConfigurationInteraction:
    def __init__(self, molecule: Molecule, molints: MolecularIntegrals):
        self.molecule = molecule
        self.molints = molints
        self.S = molints.overlap_matrix()
        self.T = molints.kinetic_matrix()
        self.V = molints.nuclear_attraction_matrix()
        self.ERI = molints.electron_repulsion_tensor(symmetrize = True)
        self.Nsite = 2 * self.S.shape[0]
        self.second_quantization()
        self.get_fermion_operators()

    def second_quantization(self):
        self.Nsite = 2 * self.S.shape[0]

        self.S_p = csc_matrix(np.array([[0, 0], [1, 0]], dtype=float))
        self.S_m = self.S_p.getH()

        self.Z = csc_matrix(np.array([[1, 0], [0, -1]], dtype=float))
        self.I = csc_matrix(np.array([[1, 0], [0, 1]], dtype=float))

        self.vacuum = eye(2**self.Nsite, format="csc")[:, 0]
    
    def get_fermion_operators(self):
        a_d = [self.S_p] + [self.I for _ in range(self.Nsite - 1)]
        self.a_dag = [a_d]

        for _ in range(self.Nsite - 1):
            a_d = self.a_dag[-1][:]
            a_d.insert(0, self.Z)
            a_d.pop()
            self.a_dag.append(a_d)
        self.c = [self.reduce_kron(ad) for ad in self.a_dag]
        self.a = [a_dag.getH() for a_dag in self.c]
        
    def reduce_kron(self, operator_list):
        result = operator_list[0]
        for operator in operator_list[1:]:
            result = kron(result, operator, format = 'csc')
        return result
    
    def get_number_operator(self):
        shape = self.c[0].shape
        self.number_operator = 0.0*eye(*shape, format = 'csc')
        for i in range(self.Nsite):
            self.number_operator += self.c[i] @ self.a[i]
            
    def get_fullci_hamiltonian(self):
        eigvals, eigvecs = np.linalg.eigh(self.S)
        X = eigvecs @ np.diag(1/np.sqrt(eigvals))
        self.h = X.T @ (self.T+self.V) @ X 
        self.v = np.einsum('ijkl, ip,jq, kr,ls -> pqrs ', self.ERI, X,X,X,X)
        shape = self.c[0].shape   
        self.hamiltonian = 0.0*eye(*shape, format = 'csc')
        for i in range(self.Nsite):
            for j in range(self.Nsite):
                ij = (i%2 == j%2)
                self.hamiltonian +=  ij * self.h[i//2,j//2] * self.c[i] @ self.a[j]
                for k in range(self.Nsite):
                    for l in range(self.Nsite):
                        kl = (k%2 == l%2)

                        self.hamiltonian += 0.5 * ij * kl* self.v[i//2,j//2,k//2,l//2] * \
                        self.c[i] @ self.c[k] @ self.a[l] @ self.a[j]
    def get_subspace(self, N):
        a_dag = combinations(self.c, N)
        shape = self.c[0].shape  
        projector = 0.0 * eye(*shape, format = 'csc')
        for op in a_dag:
            c = op[0]
            for opi in op[1:]:
                c = opi @ c
            ket = c @ self.vacuum
            bra = ket.T
            projector += ket @ bra
        return projector
            
                

In [33]:
ci = ConfigurationInteraction(H4, molints)

In [34]:
projector = ci.get_subspace(4)
ci.get_fullci_hamiltonian()
H_proj = projector @ ci.hamiltonian @ projector
fci_roots_electronic = np.sort(eigs(H_proj, k=6, which="SR", return_eigenvectors=False).real)
fci_ground_electronic = fci_roots_electronic[0]

In [35]:
fci_roots_electronic

array([-3.33038872, -3.31325788, -3.31325788, -3.31325788, -3.28994983,
       -3.28820774])

In [36]:
from pyscf import gto, scf, fci

pyscf_atoms = "; ".join(
    f"{atom.symbol} {atom.coord[0]} {atom.coord[1]} {atom.coord[2]}"
    for atom in H4.atoms
)
mol = gto.M(atom=pyscf_atoms, basis="sto-3g", unit="Angstrom")
mf = scf.RHF(mol)
mf.kernel()
pyscf_fci_total, _ = fci.FCI(mf).kernel()
pyscf_nuclear_repulsion = mol.energy_nuc()
pyscf_fci_electronic = pyscf_fci_total - pyscf_nuclear_repulsion
print(f"Notebook Full-CI electronic energy = {fci_ground_electronic}")
print(f"PySCF Full-CI electronic energy = {pyscf_fci_electronic}")
print(f"PySCF Full-CI total energy = {pyscf_fci_total}")
print(f"Nuclear repulsion = {pyscf_nuclear_repulsion}")
print(f"Difference (notebook - PySCF electronic) = {fci_ground_electronic - pyscf_fci_electronic}")

converged SCF energy = -1.54125526259767
Notebook Full-CI electronic energy = -3.3303887234623373
PySCF Full-CI electronic energy = -3.330388605150464
PySCF Full-CI total energy = -1.897849389019548
Nuclear repulsion = 1.432539216130916
Difference (notebook - PySCF electronic) = -1.1831187318733782e-07
